# Étape 3 — Modèle Avancé : Random Forest

Dans ce notebook, on entraîne un **Random Forest** sur les deux cibles du jeu de Morpion :
- `x_wins` : X gagne-t-il en jeu parfait depuis cet état ?
- `is_draw` : la partie est-elle nulle en jeu parfait depuis cet état ?

Le Random Forest est une **forêt d'arbres de décision** : au lieu d'entraîner un seul arbre (qui peut facilement sur-apprendre), on en entraîne plusieurs centaines sur des sous-ensembles aléatoires des données, puis on fait voter la majorité. Le résultat est bien plus robuste et généralement plus précis.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
print("Librairies chargées avec succès.")

ModuleNotFoundError: No module named 'pandas'

## 1. Chargement des données

In [ ]:
# Chargement du fichier CSV généré à l'Étape 0
df = pd.read_csv("../../ressources/dataset_morpion.csv")

# Séparation des features (les 18 cases) et des deux cibles
X = df.drop(columns=['x_wins', 'is_draw'])
y_wins = df['x_wins']
y_draw = df['is_draw']

# Découpage 80% entraînement / 20% test
X_train, X_test, y_train_wins, y_test_wins = train_test_split(X, y_wins, test_size=0.2, random_state=42)
_, _, y_train_draw, y_test_draw = train_test_split(X, y_draw, test_size=0.2, random_state=42)

print(f"Nombre d'échantillons d'entraînement : {X_train.shape[0]}")
print(f"Nombre d'échantillons de test         : {X_test.shape[0]}")

## 2. Baseline — Régression Logistique

On ré-entraîne rapidement la baseline ici pour pouvoir comparer les résultats directement dans ce notebook, sans avoir à consulter un autre fichier.

In [ ]:
# Entraînement de la baseline sur les deux cibles
lr_wins = LogisticRegression(max_iter=1000, random_state=42)
lr_wins.fit(X_train, y_train_wins)

lr_draw = LogisticRegression(max_iter=1000, random_state=42)
lr_draw.fit(X_train, y_train_draw)

# Métriques de la baseline
lr_pred_wins = lr_wins.predict(X_test)
lr_pred_draw = lr_draw.predict(X_test)

print("Baseline — Régression Logistique")
print(f"  Accuracy x_wins  : {accuracy_score(y_test_wins, lr_pred_wins):.2%}")
print(f"  Accuracy is_draw : {accuracy_score(y_test_draw, lr_pred_draw):.2%}")

## 3. Entraînement du Random Forest

On entraîne deux forêts indépendantes, une pour chaque cible.

**Principaux hyperparamètres :**
- `n_estimators` : nombre d'arbres dans la forêt. Plus il y en a, plus le modèle est stable — mais plus l'entraînement est lent.
- `max_depth` : profondeur maximale de chaque arbre individuel.
- `max_features='sqrt'` : à chaque nœud, seules √18 ≈ 4 features sont candidates. Cela force la diversité entre les arbres.

In [ ]:
# Modèle pour la victoire de X
rf_wins = RandomForestClassifier(
    n_estimators=200,     # Nombre d'arbres dans la forêt
    max_depth=10,         # Profondeur maximale de chaque arbre
    max_features='sqrt',  # Nombre de features candidates à chaque split
    n_jobs=-1,            # Utilise tous les cœurs disponibles pour accélérer
    random_state=42
)

print("Entraînement du Random Forest (Wins)...")
rf_wins.fit(X_train, y_train_wins)

# Modèle pour le match nul — mêmes paramètres pour cohérence
rf_draw = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42
)

print("Entraînement du Random Forest (Draws)...")
rf_draw.fit(X_train, y_train_draw)

print("\nModèles entraînés avec succès.")

## 4. Évaluation — x_wins

In [ ]:
# Prédictions sur le jeu de test
y_pred_wins = rf_wins.predict(X_test)

print("\n--- RÉSULTATS RANDOM FOREST (VICTOIRES X) ---")
print(f"Précision globale (Accuracy) : {accuracy_score(y_test_wins, y_pred_wins):.2%}")
print("\nRapport de classification :")
print(classification_report(y_test_wins, y_pred_wins))

# Matrice de confusion
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test_wins, y_pred_wins)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', cbar=False)
plt.title('Matrice de Confusion : Random Forest — x_wins')
plt.xlabel('Prédictions')
plt.ylabel('Réalité')
plt.show()

## 5. Évaluation — is_draw

In [ ]:
# Prédictions sur le jeu de test
y_pred_draw = rf_draw.predict(X_test)

print("\n--- RÉSULTATS RANDOM FOREST (MATCHS NULS) ---")
print(f"Précision globale (Accuracy) : {accuracy_score(y_test_draw, y_pred_draw):.2%}")
print("\nRapport de classification :")
print(classification_report(y_test_draw, y_pred_draw))

# Matrice de confusion
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test_draw, y_pred_draw)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', cbar=False)
plt.title('Matrice de Confusion : Random Forest — is_draw')
plt.xlabel('Prédictions')
plt.ylabel('Réalité')
plt.show()

## 6. Comparaison avec la Baseline

On compare maintenant le Random Forest à la Régression Logistique sur les deux cibles.

In [ ]:
# Tableau récapitulatif des scores
results = {
    'Modèle': ['Régression Logistique (baseline)', 'Random Forest'],
    'Accuracy x_wins': [
        accuracy_score(y_test_wins, lr_pred_wins),
        accuracy_score(y_test_wins, y_pred_wins)
    ],
    'F1 x_wins': [
        f1_score(y_test_wins, lr_pred_wins, average='weighted'),
        f1_score(y_test_wins, y_pred_wins,  average='weighted')
    ],
    'Accuracy is_draw': [
        accuracy_score(y_test_draw, lr_pred_draw),
        accuracy_score(y_test_draw, y_pred_draw)
    ],
    'F1 is_draw': [
        f1_score(y_test_draw, lr_pred_draw, average='weighted'),
        f1_score(y_test_draw, y_pred_draw,  average='weighted')
    ],
}

df_results = pd.DataFrame(results).set_index('Modèle')
print(df_results.to_string(float_format='{:.4f}'.format))

In [ ]:
# Visualisation comparative
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, cible, cols in zip(
    axes,
    ['x_wins', 'is_draw'],
    [['Accuracy x_wins', 'F1 x_wins'], ['Accuracy is_draw', 'F1 is_draw']]
):
    df_plot = df_results[cols].copy()
    df_plot.columns = ['Accuracy', 'F1']
    df_plot.T.plot(kind='bar', ax=ax, alpha=0.85, rot=0)
    ax.set_ylim(0, 1.1)
    ax.set_title(f'Comparaison — {cible}', fontweight='bold')
    ax.set_ylabel('Score')
    ax.legend(fontsize=9)
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)

plt.suptitle('Random Forest vs Baseline', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Courbe d'apprentissage — erreur OOB

L'**erreur OOB** (*Out-Of-Bag*) est une mesure d'erreur interne au Random Forest : à chaque arbre, les données non utilisées pour l'entraîner servent de mini-jeu de validation. On trace ici l'évolution de cette erreur selon le nombre d'arbres pour vérifier que 200 arbres suffisent à stabiliser le modèle.

In [ ]:
oob_errors_wins = []
oob_errors_draw = []
n_estimators_range = range(10, 210, 10)

for n in n_estimators_range:
    # oob_score=True active le calcul de l'erreur Out-Of-Bag
    clf_w = RandomForestClassifier(n_estimators=n, max_depth=10, oob_score=True, n_jobs=-1, random_state=42)
    clf_d = RandomForestClassifier(n_estimators=n, max_depth=10, oob_score=True, n_jobs=-1, random_state=42)
    clf_w.fit(X_train, y_train_wins)
    clf_d.fit(X_train, y_train_draw)
    oob_errors_wins.append(1 - clf_w.oob_score_)
    oob_errors_draw.append(1 - clf_d.oob_score_)

plt.figure(figsize=(9, 4))
plt.plot(n_estimators_range, oob_errors_wins, label='x_wins',  color='green', linewidth=2)
plt.plot(n_estimators_range, oob_errors_draw, label='is_draw', color='orange', linewidth=2)
plt.axvline(200, color='red', linestyle='--', linewidth=0.9, alpha=0.6, label='n=200 (choix final)')
plt.xlabel("Nombre d'arbres")
plt.ylabel("Erreur OOB")
plt.title("Convergence de l'erreur OOB selon le nombre d'arbres")
plt.legend()
plt.show()

## 8. Importance des features

Le Random Forest calcule une importance moyenne sur tous les arbres, ce qui la rend plus fiable que celle d'un arbre unique. On la visualise sous forme de plateau 3×3 — séparément pour les cases X (`ci_x`) et O (`ci_o`).

In [ ]:
feature_names = X.columns.tolist()

for model, title in [
    (rf_wins, 'Importance des features — Random Forest : x_wins'),
    (rf_draw, 'Importance des features — Random Forest : is_draw'),
]:
    imp = dict(zip(feature_names, model.feature_importances_))
    board_x = np.array([imp.get(f'c{i}_x', 0) for i in range(9)]).reshape(3, 3)
    board_o = np.array([imp.get(f'c{i}_o', 0) for i in range(9)]).reshape(3, 3)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, board, label, cmap in zip(
        axes,
        [board_x, board_o],
        ['Cases occupées par X (ci_x)', 'Cases occupées par O (ci_o)'],
        ['Greens', 'Blues']
    ):
        sns.heatmap(
            board, annot=True, fmt='.4f', cmap=cmap, ax=ax,
            linewidths=0.5, linecolor='gray',
            xticklabels=['Col 0', 'Col 1', 'Col 2'],
            yticklabels=['Ligne 0', 'Ligne 1', 'Ligne 2'],
            vmin=0
        )
        ax.set_title(label)

    plt.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 9. Export des modèles

On sauvegarde les deux modèles entraînés au format `.pkl` pour les réutiliser dans l'interface jouable à l'Étape 4.

In [ ]:
joblib.dump(rf_wins, '../../ressources/model_rf_wins.pkl')
joblib.dump(rf_draw, '../../ressources/model_rf_draw.pkl')

print("Fichiers .pkl générés avec succès !")
print("  → ressources/model_rf_wins.pkl")
print("  → ressources/model_rf_draw.pkl")